### **Notebook 4 (etapa 5): SHAP aplicado a los modelos estrictos de severidad y consumo de recursos**

In [ ]:
import os  # Interacción con el sistema operativo (creación de directorios y manejo de rutas)
import gc  # Recolección de basura (Garbage Collector) para liberar memoria RAM durante procesos masivos
import time  # Medición de tiempos de ejecución para monitoreo del rendimiento
import pickle  # Serialización nativa de Python para cargar el modelo XGBoost pre-entrenado
import numpy as np  # Facilita la realización de cálculos numéricos avanzados y manejo de matrices
import pandas as pd  # Permite el manejo y análisis de estructuras de datos tabulares (DataFrames)
import shap  # Biblioteca principal basada en teoría de juegos para la explicabilidad de modelos predictivos
import matplotlib.pyplot as plt  # Biblioteca para la creación y exportación de visualizaciones gráficas
import warnings  # Control de advertencias del sistema
warnings.filterwarnings("ignore", category=UserWarning)  # Suprime advertencias no críticas para mantener la consola limpia

def generar_explicabilidad_shap_estricto(target_name):
    """
    Descripción:
        Ejecuta un pipeline unificado de explicabilidad (SHAP) diseñado específicamente para el 
        Modelo Clínico Estricto (Ablación), el cual fue entrenado excluyendo variables circulares 
        propias del algoritmo GRD. Genera nuevas carpetas con el sufijo '_ESTRICTO' para evitar 
        sobrescribir el análisis de auditoría original. Extrae direccionalidad e impactos para la 
        clase de mayor riesgo y compara el escenario Global vs Oncológico.

    Entradas:
        - target_name (str): Nombre de la variable objetivo multiclase a evaluar ('SEVERIDAD' o 'CONSUMO_RECURSOS').

    Salidas:
        - None: La función no retorna variables en memoria, pero guarda en el disco local:
            1. Matrices crudas de SHAP (.npy) como respaldo de seguridad.
            2. Reportes CSV con impacto absoluto, porcentual y análisis direccional (numérico y categórico).
            3. Gráficos PNG de resumen general/categórico y paneles de dependencia.
            4. Un reporte CSV cruzando las diferencias de impacto del Top 20 entre Global vs Oncológico.
    """
    # -------------------------------------------------------------------------
    # CONFIGURACIÓN DINÁMICA POR TARGET
    # -------------------------------------------------------------------------
    # Ajustar índices y descripciones dependiendo de la variable analizada
    if target_name == 'SEVERIDAD':
        idx_clase_alta = 3  # Clase 3 es la de mayor riesgo para Severidad
        nombre_efecto_str = 'Severidad Alta (Clase 3)'
    elif target_name == 'CONSUMO_RECURSOS':
        idx_clase_alta = 2  # Clase 2 es la de mayor riesgo para Consumo
        nombre_efecto_str = 'Consumo Alto (Clase 2)'
    else:
        # Control de seguridad por si se ingresa un target inválido
        print("Target no reconocido. Ajustar índices.")
        return

    # -------------------------------------------------------------------------
    # RUTAS (Se añaden sufijos _ESTRICTO para aislar los resultados)
    # -------------------------------------------------------------------------
    # Definir el directorio de lectura de los datos de evaluación
    dir_datos = "../../Datos/Datasets Finales"
    # Definir el directorio donde se ubica el modelo estricto (Ablación) guardado previamente
    dir_modelos = "../../Resultados/Resultados (etapa 3 y 4)/XGBoost"
    
    # NUEVA CARPETA PARA LOS RESULTADOS ESTRICTOS (Evita mezclar con el modelo estándar)
    dir_base_resultados = f"../../Resultados/Resultados (etapa 5)/SHAP_{target_name}_ESTRICTO"
    os.makedirs(dir_base_resultados, exist_ok=True)
    
    # Asume que guardaste el modelo del paso de ablación con este nombre estandarizado
    nombre_modelo = f"Modelo_Optimo_XGBoost_{target_name}_ESTRICTO.pkl"
    ruta_modelo = os.path.join(dir_modelos, nombre_modelo)
    
    # Variables numéricas continuas (Se mantienen solo las que sobrevivieron a la ablación circular)
    vars_num = ['CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'EDAD']
    
    # Imprimir encabezado de ejecución en consola
    print("="*80)
    print(f"INICIANDO FASE 5: SHAP ESTRICTO (SIN FUGA) - TARGET: {target_name}")
    print(f"Foco clínico de análisis direccional: {nombre_efecto_str}")
    print("="*80)
    
    # Validar la existencia física del modelo antes de ejecutar el pipeline masivo
    if not os.path.exists(ruta_modelo):
        print(f"ERROR: No se encontró el modelo estricto en la ruta: {ruta_modelo}")
        print("Asegúrate de guardar el modelo en el script de ablación usando joblib.dump o pickle.dump")
        return
        
    print(f"-> Cargando modelo estricto desde: {nombre_modelo}...")
    # Deserializar y cargar el modelo XGBoost estricto desde el disco
    with open(ruta_modelo, 'rb') as f:
        modelo_xgb = pickle.load(f)
        
    print("-> Inicializando SHAP TreeExplainer nativo...")
    # Instanciar el motor de explicabilidad basado en árboles para el modelo cargado
    explainer = shap.TreeExplainer(modelo_xgb)
    
    # Extraer estrictamente la lista de variables que el modelo estricto usó para entrenar (ignora las circulares)
    features = modelo_xgb.get_booster().feature_names
    
    # Función interna que encapsula todo el procesamiento SHAP para un DataFrame dado
    def procesar_enfoque_shap(df_origen, tipo_enfoque, nombre_carpeta_sub):
        # Anunciar el dataset bajo análisis (Global u Onco)
        print(f"\n--- Procesando enfoque: {tipo_enfoque.upper()} (Datos: {len(df_origen)}) ---")
        
        # Crear subdirectorios de trabajo dentro de la carpeta estricta
        dir_sub_enfoque = os.path.join(dir_base_resultados, nombre_carpeta_sub)
        dir_dependence = os.path.join(dir_sub_enfoque, "Dependence_Plots")
        os.makedirs(dir_dependence, exist_ok=True)
        
        # Alinear columnas con el modelo y convertirlas a float32 para optimizar memoria en cálculos SHAP
        X_shap = df_origen[features].astype('float32')
        # Liberar dataframe madre inmediatamente
        del df_origen; gc.collect()
        
        # Registrar el tiempo de inicio
        inicio_time = time.time()
        
        # Parámetros para computar SHAP en bloques (Lotes)
        batch_size = 10000
        resultados_list = []
        n_batches = (len(X_shap) // batch_size) + (1 if len(X_shap) % batch_size != 0 else 0)
        
        # Bucle de iteración de procesamiento por lotes
        for i in range(0, len(X_shap), batch_size):
            batch = X_shap.iloc[i:i+batch_size]
            print(f"      -> Procesando bloque {i//batch_size + 1} de {n_batches}...")
            # Inferir valores SHAP para el lote
            shap_obj = explainer(batch)
            resultados_list.append(shap_obj.values)
            # Limpiar memoria del lote
            del batch, shap_obj; gc.collect()
            
        # Concatenar todos los lotes procesados en una gran matriz de impacto predictivo
        matriz_shap = np.concatenate(resultados_list, axis=0)
        print(f"   -> SHAP completado en {round((time.time() - inicio_time)/60, 2)} minutos.")
        
        # Filtro oncológico de variables constantes (sin varianza en este subgrupo)
        if "onco" in nombre_carpeta_sub.lower():
            varianzas = X_shap.var()
            cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
            # Forzar remoción del flag SIN_CANCER si persiste anómalamente
            if 'CATEGORIA_CANCER_SIN_CANCER' in X_shap.columns and 'CATEGORIA_CANCER_SIN_CANCER' not in cols_a_eliminar:
                cols_a_eliminar.append('CATEGORIA_CANCER_SIN_CANCER')
                
            # Eliminar dimensiones constantes de X_shap y del tensor de resultados SHAP
            if cols_a_eliminar:
                idx_a_eliminar = [X_shap.columns.get_loc(col) for col in cols_a_eliminar]
                X_shap = X_shap.drop(columns=cols_a_eliminar)
                matriz_shap = np.delete(matriz_shap, idx_a_eliminar, axis=1)
                print(f"      FILTRO: Se excluyeron {len(cols_a_eliminar)} variables constantes.")
        
        # Configurar clases y sufijos estandarizados
        n_clases = matriz_shap.shape[2] if len(matriz_shap.shape) == 3 else 1
        sufijo_archivo = "GLOBAL_ESTRICTO" if "global" in nombre_carpeta_sub.lower() else "ONCO_ESTRICTO"
        
        # Guardar respaldo crudo del tensor en formato Numpy (.npy)
        ruta_npy = os.path.join(dir_sub_enfoque, f"BACKUP_MATRIZ_{target_name}_{sufijo_archivo}.npy")
        np.save(ruta_npy, matriz_shap)
        
        # Exportar CSV Absoluto (Promedios sin signo)
        shap_abs = np.abs(matriz_shap).mean(axis=0) 
        if n_clases > 1:
            # En multiclase, crear la columna sumatoria 'Impacto_Total' para ordenar el peso global
            impacto_total = shap_abs.sum(axis=1)
            columnas_csv = [f"Clase_{i}" for i in range(n_clases)]
            df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=columnas_csv)
            df_shap_imp['Impacto_Total'] = impacto_total
        else:
            df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=['Impacto_Total'])
            
        # Ordenar y guardar las importancias absolutas
        df_shap_imp = df_shap_imp.sort_values(by='Impacto_Total', ascending=False)
        ruta_csv = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{target_name}_{sufijo_archivo}.csv")
        df_shap_imp.to_csv(ruta_csv, index_label='Variable')

        # Exportar CSV Porcentual (Impacto fraccionado a 100%)
        print("   -> Generando matriz de importancias porcentuales...")
        df_shap_porcentajes = df_shap_imp.copy()
        cols_num_pct = df_shap_porcentajes.select_dtypes(include=['number']).columns
        # Escalar matemáticamente cada columna numérica al 100%
        for col in cols_num_pct:
            suma_total = df_shap_porcentajes[col].sum()
            if suma_total > 0:
                df_shap_porcentajes[col] = (df_shap_porcentajes[col] / suma_total) * 100
        
        # Guardar el CSV porcentual
        ruta_csv_pct = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{target_name}_{sufijo_archivo}_PORCENTAJES.csv")
        df_shap_porcentajes.to_csv(ruta_csv_pct, index_label='Variable')
        
        # Formatear el dataframe devuelto para el proceso de comparación
        df_retorno = df_shap_porcentajes.reset_index().rename(columns={'index': 'Variable'})
        
        # Gráficos Summary Plots (Visualización Top 20)
        # 1. Summary Plot General
        plt.figure(figsize=(12, 8))
        df_top20 = df_shap_imp.head(20).drop(columns=['Impacto_Total']).iloc[::-1]
        df_top20.plot(kind='barh', stacked=True, figsize=(12, 8), cmap='viridis', ax=plt.gca())
        plt.title(f'Top 20 Variables SHAP - Enfoque {tipo_enfoque} Estricto ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_General_{target_name}_{sufijo_archivo}.png"), dpi=300)
        plt.close()
        
        # 2. Summary Plot Filtrado Solo a Categóricas
        plt.figure(figsize=(12, 8))
        vars_cat_ohe = [col for col in df_shap_imp.index if col not in vars_num]
        df_top20_cat = df_shap_imp.loc[vars_cat_ohe].head(20).drop(columns=['Impacto_Total']).iloc[::-1]
        df_top20_cat.plot(kind='barh', stacked=True, figsize=(12, 8), cmap='plasma', ax=plt.gca())
        plt.title(f'Top 20 Variables Categóricas SHAP - Enfoque {tipo_enfoque} Estricto ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_Categoricas_{target_name}_{sufijo_archivo}.png"), dpi=300)
        plt.close()

        # Aislar tensor enfocado exclusivamente en la clase riesgosa para análisis direccional
        matriz_clase_alta = matriz_shap[:, :, idx_clase_alta]
        
        # Análisis Direccional: Numerico
        print(f"   -> Extrayendo impacto direccional ({nombre_efecto_str})...")
        rangos_direccionales = []
        for v_num in vars_num:
            if v_num in X_shap.columns:
                # Segmentar distribución continua en cuartiles
                try: bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except: bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                idx_var = X_shap.columns.get_loc(v_num)
                for rango in bins_serie.cat.categories:
                    indices_rango = (bins_serie == rango)
                    n_pacientes = indices_rango.sum()
                    if n_pacientes > 0:
                        # Identificar si el promedio en este cuartil aumenta o disminuye la probabilidad de la clase alta
                        promedio_crudo = matriz_clase_alta[indices_rango, idx_var].mean()
                        if promedio_crudo > 0: efecto = "Aumenta probabilidad (+)"
                        elif promedio_crudo < 0: efecto = "Disminuye probabilidad (-)"
                        else: efecto = "Neutral"

                        # Almacenar métricas para exportación
                        rangos_direccionales.append({
                            "Variable": v_num, "Rango": str(rango), "N_Pacientes": n_pacientes,
                            f"SHAP_Crudo_{target_name}_Clase{idx_clase_alta}": promedio_crudo, "Efecto_Clinico": efecto
                        })
                        
        # Guardar CSV con impactos direccionales numéricos
        df_rangos_num = pd.DataFrame(rangos_direccionales)
        df_rangos_num.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Numericas_{target_name}_{sufijo_archivo}.csv"), index=False)

        # Análisis Direccional: Categórico
        cat_direccionales = []
        for v_cat in vars_cat_ohe:
            idx_var = X_shap.columns.get_loc(v_cat)
            # Evaluar impacto en quienes no tienen la condición (0) vs los que sí la tienen (1)
            for valor_cat in [0, 1]:
                indices_cat = (X_shap[v_cat] == valor_cat)
                n_pacientes_cat = indices_cat.sum()
                
                if n_pacientes_cat > 0:
                    promedio_crudo = matriz_clase_alta[indices_cat, idx_var].mean()
                    if promedio_crudo > 0: efecto = "Aumenta probabilidad (+)"
                    elif promedio_crudo < 0: efecto = "Disminuye probabilidad (-)"
                    else: efecto = "Neutral"

                    cat_direccionales.append({
                        "Variable": v_cat, "Condicion_OHE": valor_cat,
                        "Significado": "Presencia (1)" if valor_cat == 1 else "Ausencia (0)",
                        "N_Pacientes": n_pacientes_cat,
                        f"SHAP_Crudo_{target_name}_Clase{idx_clase_alta}": promedio_crudo,
                        "Efecto_Clinico": efecto
                    })
                    
        # Guardar CSV con impactos direccionales categóricos
        df_cat_direccional = pd.DataFrame(cat_direccionales)
        df_cat_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Categoricas_{target_name}_{sufijo_archivo}.csv"), index=False)

        # Paneles de Dependencia (Top 20 variables)
        print(f"   -> Generando paneles de dependencia en carpeta...")
        top_20_vars = df_shap_imp.head(20).index.tolist()
        for var in top_20_vars:
            if var in X_shap.columns:
                # Crear un panel de gráficos (un plot de dependencia por cada clase predecida)
                fig, axes = plt.subplots(1, n_clases, figsize=(5 * n_clases, 4.5))
                for clase in range(n_clases):
                    valores_sh_clase = matriz_shap[:, :, clase]
                    shap.dependence_plot(var, valores_sh_clase, X_shap, interaction_index=None, ax=axes[clase], show=False)
                    axes[clase].set_title(f'Impacto en clase {clase}', fontsize=10)
                
                # Configurar títulos y guardar el panel renderizado
                fig.suptitle(f'Dependence Plot: {var} ({target_name} Estricto)', fontsize=12, y=1.02)
                plt.tight_layout()
                plt.savefig(os.path.join(dir_dependence, f"SHAP_Dependence_{var}.png"), dpi=200, bbox_inches='tight')
                plt.close()
                
        # Limpieza masiva de memoria finalizado el proceso del enfoque actual
        print(f"   Liberando memoria asignada al enfoque {tipo_enfoque}...")
        del X_shap, matriz_shap, df_shap_imp, df_shap_porcentajes
        gc.collect()
        
        return df_retorno

    # -------------------------------------------------------------------------
    # EJECUCIÓN SECUENCIAL Y REPORTE CRUZADO (DIFERENCIAS)
    # -------------------------------------------------------------------------
    print("\n--- PASO A: Cargando datos para análisis global ---")
    # Cargar test oncológico y test control puros
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control.csv"), low_memory=False)
    
    # Calcular y muestrear población de control manteniendo prevalencia del target para lograr 200,000 pacientes totales
    n_onco_total = len(df_onco_test)
    n_control_needed = 200000 - n_onco_total 
    
    proporciones_control = df_control_test[target_name].value_counts(normalize=True)
    df_ctrl_sample = df_control_test.groupby(target_name, group_keys=False).apply(
        lambda x: x.sample(min(len(x), int(np.round(n_control_needed * proporciones_control[x.name]))), random_state=42)
    )
    
    # Armar dataframe global definitivo y mezclar
    df_test_global = pd.concat([df_onco_test, df_ctrl_sample], ignore_index=True).sample(frac=1, random_state=42)
    # Limpiar datasets transitorios
    del df_control_test, df_ctrl_sample; gc.collect() 
    
    # Procesar explicabilidad modelo Estricto para el dataset Global
    df_global_pct = procesar_enfoque_shap(df_test_global, "Global_Estricto", "Valores SHAP (global)")
    
    print("\n--- PASO B: Cargando datos para análisis oncológico ---")
    # Procesar explicabilidad modelo Estricto para el dataset Oncológico
    df_onco_pct = procesar_enfoque_shap(df_onco_test, "Onco_Estricto", "Valores SHAP (oncologicos)")
    
    print("\n--- PASO C: Generando reporte comparativo ---")
    # Agregar columnas informativas con las posiciones en el ranking (Top)
    df_global_pct['Posicion_Global'] = df_global_pct.index + 1
    df_onco_pct['Posicion_Onco'] = df_onco_pct.index + 1
    
    # Aislar a los 20 predictores líderes de la cohorte Oncológica Estricta
    top_20_onco = df_onco_pct.head(20).copy()
    # Cruzar con el Global usando Left Join
    df_comparacion = pd.merge(top_20_onco, df_global_pct, on='Variable', suffixes=('_Onco', '_Global'), how='left')
    # Extraer delta (diferencia) de impacto general porcentual
    df_comparacion['Diferencia_Impacto_Total'] = df_comparacion['Impacto_Total_Onco'] - df_comparacion['Impacto_Total_Global']
    
    # Ordenar y guardar el dataset final cruzado
    columnas_finales = ['Variable', 'Posicion_Onco', 'Posicion_Global', 'Impacto_Total_Onco', 'Impacto_Total_Global', 'Diferencia_Impacto_Total']
    df_final = df_comparacion[columnas_finales]
    ruta_diferencias = os.path.join(dir_base_resultados, f"Diferencias_SHAP_{target_name}_ESTRICTO_ONCO_GLOBAL.csv")
    df_final.to_csv(ruta_diferencias, index=False)
    
    # Mensaje final de éxito
    print("\n" + "="*80)
    print("PROCESO UNIFICADO ESTRICTO FINALIZADO CON ÉXITO")
    print(f"Reportes guardados en: {dir_base_resultados}")
    print("="*80)

# EJECUCIÓN PARA LOS MODELOS ESTRICTOS
# generar_explicabilidad_shap_estricto('SEVERIDAD')
# generar_explicabilidad_shap_estricto('CONSUMO_RECURSOS')

In [4]:
generar_explicabilidad_shap_estricto('SEVERIDAD')

INICIANDO FASE 5: SHAP ESTRICTO (SIN FUGA) - TARGET: SEVERIDAD
Foco clínico de análisis direccional: Severidad Alta (Clase 3)
-> Cargando modelo estricto desde: Modelo_Optimo_XGBoost_SEVERIDAD_ESTRICTO.pkl...
-> Inicializando SHAP TreeExplainer nativo...

--- PASO A: Cargando datos para análisis global ---

--- Procesando enfoque: GLOBAL_ESTRICTO (Datos: 200000) ---
      -> Procesando bloque 1 de 20...
      -> Procesando bloque 2 de 20...
      -> Procesando bloque 3 de 20...
      -> Procesando bloque 4 de 20...
      -> Procesando bloque 5 de 20...
      -> Procesando bloque 6 de 20...
      -> Procesando bloque 7 de 20...
      -> Procesando bloque 8 de 20...
      -> Procesando bloque 9 de 20...
      -> Procesando bloque 10 de 20...
      -> Procesando bloque 11 de 20...
      -> Procesando bloque 12 de 20...
      -> Procesando bloque 13 de 20...
      -> Procesando bloque 14 de 20...
      -> Procesando bloque 15 de 20...
      -> Procesando bloque 16 de 20...
      -> Procesa

In [5]:
generar_explicabilidad_shap_estricto('CONSUMO_RECURSOS')

INICIANDO FASE 5: SHAP ESTRICTO (SIN FUGA) - TARGET: CONSUMO_RECURSOS
Foco clínico de análisis direccional: Consumo Alto (Clase 2)
-> Cargando modelo estricto desde: Modelo_Optimo_XGBoost_CONSUMO_RECURSOS_ESTRICTO.pkl...
-> Inicializando SHAP TreeExplainer nativo...

--- PASO A: Cargando datos para análisis global ---

--- Procesando enfoque: GLOBAL_ESTRICTO (Datos: 200000) ---
      -> Procesando bloque 1 de 20...
      -> Procesando bloque 2 de 20...
      -> Procesando bloque 3 de 20...
      -> Procesando bloque 4 de 20...
      -> Procesando bloque 5 de 20...
      -> Procesando bloque 6 de 20...
      -> Procesando bloque 7 de 20...
      -> Procesando bloque 8 de 20...
      -> Procesando bloque 9 de 20...
      -> Procesando bloque 10 de 20...
      -> Procesando bloque 11 de 20...
      -> Procesando bloque 12 de 20...
      -> Procesando bloque 13 de 20...
      -> Procesando bloque 14 de 20...
      -> Procesando bloque 15 de 20...
      -> Procesando bloque 16 de 20...
    